# 09 - Check of the ESM25GoodMP dataset

ESM25 on SeisBench has four datasets: `ESM25GoodCV` and `ESM25BadCV` (acceleration converted to physical units, the
data of every run so far), `ESM25GoodMP` (manually processed acceleration, velocity and displacement, good records
only) and `ESM25SpectraMP` (response spectra). A classifier trained on processed good records against unprocessed bad
records could separate the classes by the processing (filter band, taper, trimming) rather than by the quality: a
possible explanation of the AUC of 0.9999 of Shalabi et al. (2026).

Before downloading ESM25GoodMP, this notebook checks, without reading any waveform:

1. the size of the waveform files on the server, against the free disk space;
2. the metadata of ESM25GoodMP (downloaded to `data/goodmp/`);
3. how its records match those of ESM25GoodCV (identifiers in both, duplicates) and which columns are specific to the
   processed data (filter corners, processed time window, padding, sampling).

## 0. Setup

In [1]:
from dataclasses import replace
from pathlib import Path
import shutil

import pandas as pd

from wqu.config import load_config
from wqu.data.loading import load_esm25_metadata
from wqu.data.remote_datasets import (
    columns_only_in,
    download_file,
    key_overlap,
    matching_columns,
    remote_dataset_url,
    remote_file_size,
    shared_column_agreement,
)

CONFIG_PATH = Path("..") / "configs" / "default.yaml"
FULL_SOURCE = "seisbench"
MP_REPOSITORY = "esm25goodmp"
WAVEFORM_FILE = "waveforms.hdf5"
BYTES_PER_GB = 1e9
# Candidate record identifiers shared by the datasets (trace_name may refer to the position inside each file).
KEY_CANDIDATES = ["trace_name_old", "trace_name"]
# Column names that may describe the processing.
PROCESSING_PATTERNS = ["_mp", "filter", "lowcut", "highcut", "corner", "pass", "taper", "padding", "start_time",
                       "end_time", "sampling", "npts", "processing", "baseline"]
# Columns of the trace compared between the two datasets on the matched records.
COMPARED_PATTERNS = ["^trace_start_time$", "^trace_sampling_rate_hz$", "^trace_dt_s$", "^trace_has_padding",
                     "^trace_npts", "^station_code$", "^source_esm_id$"]

config = load_config(CONFIG_PATH)
config = replace(config, data=replace(config.data, source=FULL_SOURCE))
columns = config.columns
template = config.waveforms.remote_url_template
output_dir = config.preprocessing.output_dir.parent / "goodmp"
repositories = {"good MP": MP_REPOSITORY, **{f"{subset.name} CV": subset.repository_name for subset in config.data.subsets}}
print(f"Repositories: {repositories}")
print(f"Output: {output_dir}")

Repositories: {'good MP': 'esm25goodmp', 'good CV': 'esm25goodcv', 'bad CV': 'esm25badcv'}
Output: /Users/giulianaparadiso/Desktop/INGV Projects/waveform-quality-uncertainty/data/goodmp


/Users/giulianaparadiso/miniconda3/envs/wqu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Size of the files on the server

Free space measured on the disk of the home folder (where the SeisBench cache usually is) and of the project.

In [2]:
sizes = pd.DataFrame({
    name: {file_name: remote_file_size(remote_dataset_url(template, repository, file_name)) / BYTES_PER_GB
           for file_name in (WAVEFORM_FILE, config.data.seisbench_metadata_file)}
    for name, repository in repositories.items()
}).T.rename(columns=lambda name: f"{name} (GB)")
display(sizes.round(2))
for place in (Path.home(), output_dir.parent.resolve()):
    print(f"Free space on the disk of {place}: {shutil.disk_usage(place).free / BYTES_PER_GB:.1f} GB")

,waveforms.hdf5 (GB),metadata.csv (GB)
good MP,194.18,0.39
good CV,64.73,0.39
bad CV,36.32,0.03


Free space on the disk of /Users/giulianaparadiso: 34.0 GB
Free space on the disk of /Users/giulianaparadiso/Desktop/INGV Projects/waveform-quality-uncertainty/data: 34.0 GB


## 2. Metadata of ESM25GoodMP

In [3]:
mp_metadata_path = output_dir / config.data.seisbench_metadata_file
if not mp_metadata_path.exists():
    download_file(remote_dataset_url(template, MP_REPOSITORY, config.data.seisbench_metadata_file), mp_metadata_path)
text_columns = {column: str for column in columns.text}
mp_metadata = pd.read_csv(mp_metadata_path, dtype=text_columns, low_memory=False)
print(f"ESM25GoodMP: {len(mp_metadata):,} records, {mp_metadata.shape[1]} columns")
mp_metadata.head(3).T.head(40)

metadata.csv: 100%|██████████| 385M/385M [00:46<00:00, 8.24MB/s] 


ESM25GoodMP: 64,213 records, 598 columns


,0,1,2
source_esm_id,AL-2014-0005,AL-2014-0005,AL-2014-0005
source_esm_origin_time,2014-01-20T14:40:48,2014-01-20T14:40:48,2014-01-20T14:40:48
source_isc_id,609700745.0,609700745.0,609700745.0
source_usgs_id,usb000m4t2,usb000m4t2,usb000m4t2
source_ingv_id,2986911.0,2986911.0,2986911.0
source_emsc_id,20140120_0000089,20140120_0000089,20140120_0000089
source_nation_code,AL,AL,AL
source_latitude_deg,41.38,41.38,41.38
source_longitude_deg,19.45,19.45,19.45
source_depth_km,8.0,8.0,8.0


## 3. Records and columns compared with ESM25GoodCV

Identifiers in both datasets for each candidate key: the key with all the records in both and no duplicates is the one
to use to pair the processed and unprocessed version of each record.

In [4]:
_, metadata = load_esm25_metadata(config)
good_cv = metadata[metadata[columns.label] == 0].reset_index(drop=True)
overlap = pd.DataFrame({
    key: key_overlap(mp_metadata[key], good_cv[key], ("good MP", "good CV"))
    for key in KEY_CANDIDATES if key in mp_metadata.columns and key in good_cv.columns
})
overlap

,trace_name_old,trace_name
records in good MP,64213,64213
records in good CV,64213,64213
identifiers in both,64213,64213
only in good MP,0,0
only in good CV,0,0
duplicated in good MP,0,0
duplicated in good CV,0,0


In [5]:
only_mp = columns_only_in(mp_metadata, good_cv)
only_cv = columns_only_in(good_cv, mp_metadata)
print(f"Columns only in good MP ({len(only_mp)}): {only_mp}")
print(f"Columns only in good CV ({len(only_cv)}): {only_cv}")
processing_columns = matching_columns(mp_metadata.columns, PROCESSING_PATTERNS)
print(f"Columns of good MP that may describe the processing ({len(processing_columns)}):")
mp_metadata[processing_columns].describe(include="all").T

Columns only in good MP (0): []
Columns only in good CV (13): ['preferred_magnitude', 'preferred_magnitude_type', 'original_duration_s', 'recording_span_s', 'padding_fraction', 'component_start_spread_s', 'original_sampling_rate_hz', 'origin_year', 'max_unprocessed_pga_cps2', 'station_id', 'subset_index', 'subset', 'label']
Columns of good MP that may describe the processing (23):


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
station_preferred_vs30_mps,47117.0,NaN,NaN,NaN,594.640414,287.155324,90.0,397.0,514.0,753.0,3019.0
station_vs30_mps,12248.0,NaN,NaN,NaN,565.783067,335.163032,90.0,321.0,486.0,705.0,2925.0
station_vs30_mps_wa,61626.0,NaN,NaN,NaN,632.248893,281.037604,100.0,407.0,573.0,823.2,3019.0
trace_processing_status,64213,1,processed,64213,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trace_processing_type,64213,1,manual (Paolucci et al 2011),64213,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trace_npts_cv_u,64213.0,NaN,NaN,NaN,36246.267173,18154.552977,469.0,26250.0,30634.0,53140.0,176651.0
trace_npts_cv_v,64213.0,NaN,NaN,NaN,36245.132092,18154.328903,469.0,26256.0,30623.0,53142.0,176416.0
trace_npts_cv_w,64213.0,NaN,NaN,NaN,36239.197701,18150.033081,469.0,26248.0,30639.0,53134.0,176694.0
trace_has_padding_cv,64213,1,True,64213,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trace_sampling_rate_hz,64213.0,NaN,NaN,NaN,200.0,0.0,200.0,200.0,200.0,200.0,200.0


Agreement of the trace columns present in both datasets, on the records matched by the chosen key: equal start
time, sampling rate and padding flags mean that the processed traces are stored on the same time axis as the
unprocessed ones.

In [6]:
key = overlap.loc["identifiers in both"].idxmax()
shared = [column for column in matching_columns(mp_metadata.columns, COMPARED_PATTERNS) if column in good_cv.columns]
print(f"Key: {key}; compared columns: {shared}")
shared_column_agreement(mp_metadata, good_cv, key, shared).round(4)

Key: trace_name_old; compared columns: ['source_esm_id', 'station_code', 'trace_npts_cv_u', 'trace_npts_cv_v', 'trace_npts_cv_w', 'trace_has_padding_cv', 'trace_sampling_rate_hz', 'trace_dt_s', 'trace_start_time', 'trace_has_padding_mp', 'trace_npts_mp']


,matched_records,share_equal
source_esm_id,64213,1.0
station_code,64213,1.0
trace_npts_cv_u,64213,1.0
trace_npts_cv_v,64213,1.0
trace_npts_cv_w,64213,1.0
trace_has_padding_cv,64213,1.0
trace_sampling_rate_hz,64213,1.0
trace_dt_s,64213,1.0
trace_start_time,64213,1.0
trace_has_padding_mp,64213,1.0
